# Before a fault: do wind-turbine SCADA signals look different?

A worked analysis with tessa on real data: raw per-asset parquet files plus an
Excel label sheet, taken to a verdict ("are the classes separable at all?"), a
feature ranking, and a shareable HTML report.

**The example data** is Wind Farm A of
[CARE to Compare](https://doi.org/10.3390/data9120138), a public benchmark for
early fault detection: 5 onshore wind turbines with 10-minute SCADA data (54
sensors) and 22 labelled events. 12 events are the days leading up to a fault
logged by the operator (gearbox, generator bearing, hydraulic group,
transformer); the other 10 are periods of normal operation. The question:
**do the SCADA signals tell the run-up to a fault apart from normal operation,
and through which signals?**

**How to use it.** As shipped, it downloads the example data once (about 160 MB
from Zenodo, typically a minute or two) and runs end to end. To run it on your data, fill in
every `TODO` in the settings cell and set `USE_EXAMPLE_DATA = False`. Sections 8
and 9 hold the decisions you make after seeing the first results (which columns
leak the label, which confounders to correct for); they are marked `TODO` there.

1. Settings
2. The data store: which assets, sources and periods exist
3. The label sheet
4. Matching labels to data: drop events the store cannot serve
5. A look at raw events
6. Features: one row per event
7. First pass: is there any difference at all?
8. Checking the answer: leaks and confounders
9. The corrected analysis
10. Save, report, dashboard

*Data license:* the example data is CC BY-SA 4.0 (Gück, Roelofs & Faulstich,
"CARE to Compare", *Data* 9(12):138, 2024). Cite the paper if you publish
results obtained from it.

## 1. Settings

In [ ]:
from pathlib import Path

USE_EXAMPLE_DATA = True  # TODO: set to False once the paths below point at your data

# Notebooks run with the notebook's folder as working directory.
PROJECT_ROOT = Path.cwd().parent

# --- Data store ---------------------------------------------------------------
DATA_ROOT = Path("/path/to/data")  # TODO: folder with one sub-folder per asset
TIMESTAMP_COL = "timestamp"  # TODO: name of the time column inside the parquet files

# --- Label sheet --------------------------------------------------------------
LABELS_XLSX = Path("/path/to/labels.xlsx")  # TODO
LABELS_SHEET = 0  # TODO: sheet name, or 0-based sheet index
COLUMN_MAP = {  # TODO: your Excel headers -> the names tessa expects
    "Asset": "asset_id",
    "Start time": "start",
    "End time": "end",
    "Label": "class",
    "Replacement type": "replacement_type",  # optional extra column, passed through
}

# --- The question -------------------------------------------------------------
CLASSES = ["TP", "FP"]  # TODO: the classes to compare
STRATIFY_BY = None  # TODO: a label column to split the analysis by, e.g. "replacement_type"
SIGNALS = None  # TODO: e.g. ["temperature", "vibration"]; None = every column in the files
NOT_SIGNALS = []  # TODO: columns in the files that are not measurements to aggregate
PLOT_SIGNAL = None  # TODO: the signal to eyeball in section 5; None = the first one
MIN_COVERAGE = 0.9  # drop events whose window the data files cover by less than 90%

# --- Memory -------------------------------------------------------------------
# Events are read and reduced in batches, so memory follows the batch size, not
# the number of events. Lower these if a large store (hundreds of assets, high
# sampling rates, many derived features) still runs out of memory.
EVENTS_PER_BATCH = None  # events processed at once; None = one per CPU thread
MEMORY_BUDGET = "auto"  # cap on a batch's estimated memory: "auto" (half the RAM), "8GB", None

OUTPUT_DIR = PROJECT_ROOT / "outputs"
RANDOM_STATE = 7

if USE_EXAMPLE_DATA:
    # CARE to Compare, Wind Farm A (CC BY-SA 4.0): about 160 MB from Zenodo on the
    # first run, converted to the layout tessa expects and reused afterwards.
    from tessa.examples import fetch_wind_farm_a

    # A folder of its own: tessa reads every folder under a data root as an asset.
    example = fetch_wind_farm_a(PROJECT_ROOT / "example_data" / "wind_farm_a")
    DATA_ROOT, LABELS_XLSX, COLUMN_MAP = example.data_root, example.labels_path, example.column_map
    LABELS_SHEET = "events"
    CLASSES = ["anomaly", "normal"]
    # Wind and nacelle directions are compass bearings: their mean is not a level.
    # (status_type_id stays in on purpose; section 8 shows why it must go.)
    NOT_SIGNALS = example.signals.filter(example.signals["is_angle"])["column"].to_list()
    PLOT_SIGNAL = "ambient_temp_avg"

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from IPython.display import display

import tessa
from tessa import Config, Dataset, Run
from tessa.dataset import file_catalog
from tessa.features import FeatureRegistry, feature, to_period
# Importing builtins also registers the stock aggregators (mean, std, p05, ...)
from tessa.features.builtins import make_first_difference, make_rolling_std
from tessa.labels import ExcelLabelSource

pl.Config.set_tbl_rows(25)
print("tessa", tessa.__version__)

## 2. The data store

`Config` describes the on-disk layout. The defaults expect
`<DATA_ROOT>/<asset_id>/<source>_<start>_<end>.parquet`, with dates as `YYMMDD`
or `YYYYMMDD`. Files sharing a source name are concatenated over time; different
sources are outer-joined on the timestamp. If your files are laid out
differently, set `asset_subdir` or `filename_pattern` (a regex with named groups
`source`, `start`, `end`).

The catalogue below is built from file *names* only, so it is instant even on a
large store: it shows what exists before anything is read.

In [ ]:
cfg = Config(
    data_root=DATA_ROOT,
    timestamp_col=TIMESTAMP_COL,
    output_dir=OUTPUT_DIR,
    random_state=RANDOM_STATE,
    events_per_batch=EVENTS_PER_BATCH,
    memory_budget=MEMORY_BUDGET,
    # asset_subdir="input",  # files in <asset>/input/ instead of <asset>/
    # filename_pattern=r"^(?P<source>.+)_(?P<start>\d{8})_(?P<end>\d{8})\.parquet$",
)
ds = Dataset(cfg)

catalog = file_catalog(cfg)
print(f"{len(ds.assets)} asset folders, {catalog.height} parquet files under {DATA_ROOT}")
catalog.group_by("asset_id", "source").agg(
    n_files=pl.len(),
    first_day=pl.col("start").min(),
    last_day=pl.col("end").max(),
).sort("asset_id", "source")

In [ ]:
# The columns of one asset, across all its sources (schemas only, no data read)
asset = catalog["asset_id"][0]
raw_columns = ds.channels(asset)
signals = SIGNALS or [c for c in raw_columns if c != cfg.timestamp_col]
signals = [s for s in signals if s not in NOT_SIGNALS]
print(f"{asset}: {len(raw_columns)} columns, {len(signals)} signals analysed")
if USE_EXAMPLE_DATA:
    # What each column measures, from the dataset's feature description
    display(example.signals.select("column", "description", "unit").head(12))

## 3. The label sheet

`ExcelLabelSource` reads the sheet, renames columns via `COLUMN_MAP` and parses
`start` / `end` as datetimes. Every other column is kept as event metadata.

In [ ]:
labels = ExcelLabelSource(LABELS_XLSX, sheet=LABELS_SHEET, column_map=COLUMN_MAP).load(cfg)

# Extra label columns travel with every event as label metadata, whatever their
# dtype: a numeric one (a severity score, an ID) is carried into the feature table
# as-is and is never aggregated or used as a feature (see sections 6 and 7).
extras = [c for c in labels.columns if c not in (cfg.asset_col, "start", "end", cfg.class_col)]

print(f"{labels.height} label rows, extra columns: {extras}")
labels.group_by(cfg.class_col).agg(
    events=pl.len(),
    assets=pl.col(cfg.asset_col).n_unique(),
    median_days=((pl.col("end") - pl.col("start")).dt.total_hours() / 24).median(),
).sort(cfg.class_col)

In [ ]:
# Events on the same asset that overlap in time count the same data twice
pairs = labels.with_row_index("row").join(labels.with_row_index("row"), on=cfg.asset_col)
pairs.filter(
    (pl.col("row") < pl.col("row_right"))
    & (pl.col("start") <= pl.col("end_right"))
    & (pl.col("start_right") <= pl.col("end"))
).select(cfg.asset_col, "start", "end", cfg.class_col, "start_right", "end_right", f"{cfg.class_col}_right")

In the example, the two overlapping events on WT21 are one incident reported in
two CARE datasets (gearbox bearings damaged / gearbox failure). Both stay in:
the cross-validation below holds out whole turbines, so the duplicate cannot
leak from training into testing, and dropping either would be a choice the
data does not make for us.

## 4. Matching labels to data

Label sheets and data exports are maintained separately, so some rows usually
point at assets that were never exported or periods outside the export. Building
events for them would fail, so first ask the store what it can serve.

`check="rows"` also reads the timestamp column of the matching files and confirms
the window holds data. `check="files"` decides from file names alone: faster on
a big store, but it misses gaps inside a file.

In [ ]:
availability = ds.availability(labels, check="rows")
availability.group_by("data_reason").len().sort("len", descending=True)

In [ ]:
# The rows that will be dropped, and why
availability.filter(
    ~pl.col("data_available") | (pl.col("data_coverage") < MIN_COVERAGE)
).select(cfg.asset_col, "start", "end", cfg.class_col, "data_coverage", "data_reason")

In [ ]:
# Keep what the store can serve, then restrict to the classes being compared.
# A warning lists anything dropped, so the label table never shrinks silently.
labels_ok = ds.available(labels, check="rows", min_coverage=MIN_COVERAGE)
labels_ok = labels_ok.filter(pl.col(cfg.class_col).is_in(CLASSES))

print(f"{labels_ok.height} of {labels.height} label rows kept")
by = [cfg.class_col, STRATIFY_BY] if STRATIFY_BY else [cfg.class_col]
labels_ok.group_by(by).len().sort(by)

## 5. A look at raw events

`ds.events()` returns one lazy frame per event, with the label columns attached.
Nothing is read until a frame is collected. Plot a few raw events per class
before trusting any statistic.

In [ ]:
events = ds.events(labels_ok, columns=signals)  # {event_id: LazyFrame}, in label-row order
event_index = labels_ok.with_columns(event_id=pl.Series(list(events)))

plot_signal = PLOT_SIGNAL or signals[0]
N_PER_CLASS = 5
span = (labels_ok["end"] - labels_ok["start"]).median()
unit, per = ("days", 86400) if span.days >= 2 else ("hours", 3600)

fig, axes = plt.subplots(1, len(CLASSES), figsize=(5 * len(CLASSES), 3.5), sharey=True)
for ax, cls in zip(np.atleast_1d(axes), CLASSES):
    ids = event_index.filter(pl.col(cfg.class_col) == cls)["event_id"].head(N_PER_CLASS)
    for event_id in ids:
        df = events[event_id].select(cfg.timestamp_col, plot_signal).drop_nulls().collect()
        elapsed = (df[cfg.timestamp_col] - df[cfg.timestamp_col][0]).dt.total_seconds() / per
        ax.plot(elapsed, df[plot_signal], linewidth=0.6)
    ax.set_title(f"{cls}: {len(ids)} events")
    ax.set_xlabel(f"{unit} since event start")
np.atleast_1d(axes)[0].set_ylabel(plot_signal)
fig.tight_layout()

In the example, the ambient temperature already hints at something: the fault
run-ups are mostly in summer and autumn, the normal periods more often in
winter and spring. Keep that in mind for section 8.

## 6. Features: one row per event

Each signal is aggregated over the event window: mean, standard deviation, and
the 5th and 95th percentiles. One row per event is the honest unit here: the
events, not the 10-minute samples, are the independent observations.

To also see *dynamics*, derive per-sample features before aggregating, such as
the step between consecutive samples (`make_first_difference`) or a rolling
standard deviation (`make_rolling_std`), or write your own with `@feature`
(section 9 does). Features multiply the table's width by the number of
aggregators, so add them for a reason.

`to_period` reads the events in batches (`EVENTS_PER_BATCH`, `MEMORY_BUDGET`),
so a store of hundreds of assets does not need to fit in memory at once.

In [ ]:
AGGREGATORS = ["mean", "std", "p05", "p95"]

features = FeatureRegistry()  # a fresh registry, so re-running this cell is safe
# for s in signals:  # dynamics, if your question is about them:
#     make_first_difference(s, registry=features)
#     make_rolling_std(s, window=10, registry=features)  # window in samples

period = to_period(events, cfg, aggregators=AGGREGATORS, feature_registry=features)
print(f"{period.height} events x {period.width} columns")
period.head()

## 7. First pass: is there any difference at all?

`Run` holds the table and the question. Cross-validation folds are grouped by
asset, so a model is always scored on assets it never saw in training: an asset
signature cannot pass for a class signature.

**Separability first.** It answers "can *any* model tell the classes apart
better than chance?" with a permutation test. If the verdict is "not separable",
a feature ranking would rank noise. With 22 events, 100 permutations take a few
tens of seconds; raise them for a final version.

In [ ]:
SEPARABILITY = dict(
    n_permutations=100,  # the smallest reachable p-value is about 1 / (n + 1)
    rf_params=dict(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE),
)


def make_run(table: pl.DataFrame) -> Run:
    return Run(
        table,
        target_col=cfg.class_col,
        cfg=cfg,
        label_filter={cfg.class_col: CLASSES},
        # Label-sheet columns stay in the table for filtering and stratifying, but
        # never become features, even when numeric: they would leak the label.
        label_cols=events.label_cols,
    )


first = make_run(period)
first.separability(**SEPARABILITY).frames["summary"]

In [ ]:
# Which features carry the difference: a blend of model-based and univariate rankings
first_ranking = first.importance().frames["table"]
first_ranking.head(10)

## 8. Checking the answer: leaks and confounders

A verdict is only as good as the table behind it. Two checks before believing
it:

- **Leaks.** Is a column in the data derived from the labels themselves? The
  symptom is a score that looks too good, or a top feature that is too clean.
- **Confounders.** Do the classes differ in something outside the question
  (season, load, operating mode, asset age) that the model can learn instead?

### 8.1 Leaks

Look at what the top-ranked features *are*, per class.

In [ ]:
top = list(first_ranking.index[:6])
period.group_by(cfg.class_col).agg(pl.col(top).mean()).sort(cfg.class_col)

In the example, the three top features are `status_type_id`: the turbine's
status code averages about 4 ("downtime") in the fault run-ups and is 0 ("normal
operation") in every normal period. Yet the turbines keep producing power during
those "downtimes": in Wind Farm A the status was derived from the same fault
logbook as the labels. It is not a measurement but the label in disguise, and it
has to go. (The dataset authors use it to pick clean training data, which is
legitimate; using it as a feature is not.)

In [ ]:
LEAKS = ["status_type_id"] if USE_EXAMPLE_DATA else []  # TODO: columns that leak the label

leak_cols = [c for c in period.columns if c.split("__")[0] in LEAKS]
no_leak = make_run(period.drop(leak_cols))
no_leak.separability(**SEPARABILITY).frames["summary"]

In [ ]:
no_leak_ranking = no_leak.importance().frames["table"]
no_leak_ranking.head(10)

### 8.2 Confounders

Without the leak the classes still look separable, but in the example the top
features are the *cold-end* (5th percentile) temperatures of cabinets, the
nacelle and the nose cone: things that follow the weather, not a failing
component. The fault run-ups are mostly in summer and autumn. If the season
alone separates the classes as well as all the signals together, the
"difference" is the calendar.

In [ ]:
CONFOUNDERS = ["ambient_temp_avg"] if USE_EXAMPLE_DATA else []  # TODO: e.g. season, load

confounder_cols = [c for c in period.columns if c.split("__")[0] in CONFOUNDERS]
label_columns = [c for c in period.columns if c in events.label_cols]
if confounder_cols:
    confounder_only = make_run(period.select(label_columns + confounder_cols))
    display(confounder_only.separability(**SEPARABILITY).frames["summary"])

In the example, the ambient temperature alone reaches a balanced accuracy of
about 0.72 (p ≈ 0.04), close to the 0.80 of all 74 signals and with the same
p-value. Most of the remaining difference is the season.

## 9. The corrected analysis

Remove the confounder from the representation rather than only from the table:
a component's temperature *rise above ambient* is what a failing bearing or
cooler changes, while the absolute temperature mostly follows the weather. The
`@feature` decorator adds such a derived signal; it is computed per 10-minute
sample, then aggregated like any other.

For your data, the right correction depends on the confounder: subtract or
divide by a reference signal, keep only one operating mode, or compare each
event to the same asset's own history.

In [ ]:
# TODO: the reference signal, the signals to express relative to it, a name suffix
if USE_EXAMPLE_DATA:
    REFERENCE, SUFFIX = "ambient_temp_avg", "over_ambient"
    is_temperature = example.signals["unit"] == "°C"
    RELATIVE = [c for c in example.signals.filter(is_temperature)["column"] if c in signals]
    RELATIVE.remove(REFERENCE)
else:
    REFERENCE, SUFFIX, RELATIVE = None, "rel", []

corrected_features = FeatureRegistry()
for s in RELATIVE:

    @feature(f"{s}__{SUFFIX}", deps=(s, REFERENCE), registry=corrected_features)
    def _(s=s):
        return pl.col(s) - pl.col(REFERENCE)


# Aggregate the relative signals instead of the absolute ones, without the
# confounder itself and without the leaks.
dropped = set(RELATIVE) | set(CONFOUNDERS) | set(LEAKS)
sources = [s for s in signals if s not in dropped] + corrected_features.names()
corrected = to_period(
    events, cfg, sources=sources, aggregators=AGGREGATORS, feature_registry=corrected_features
)
print(f"{corrected.height} events x {corrected.width} columns")

In [ ]:
run = make_run(corrected)
separability = run.separability(**SEPARABILITY)
separability.frames["summary"]

In [ ]:
importance = run.importance()
importance.frames["table"].head(10)

In [ ]:
# Out-of-fold classifier: how well, and which classes get confused
cv = run.cv_classifier()
print(cv.summary(max_rows=5))

**What the example says.** Once the leak is gone and temperatures are measured
against the weather, the evidence is weak: in our run the balanced accuracy is
about 0.65 with a permutation p-value of about 0.18, "not separable". With 22
events and faults in four different subsystems there is no *common* signature of
an approaching fault in these 10-minute averages. Single events can still show
one: before the generator-bearing failure on WT10, the non-drive-end bearing ran
about 50 °C above ambient on average, more than in any other event (16 to 42 °C).
That is the honest answer to the question as asked, and it matches how the
benchmark is meant to be used: a model of each turbine's *own* normal behaviour,
judged fault by fault.

The first pass looked like a finding (separable, p ≈ 0.02), and so did the
second without the leak (p ≈ 0.04). It was a leak plus the season.

In [ ]:
# Does the answer hold within each stratum? (Too few events in a stratum -> no verdict.)
if STRATIFY_BY:
    for (value,), part in corrected.group_by(STRATIFY_BY, maintain_order=True):
        counts = part[cfg.class_col].value_counts().sort(cfg.class_col).rows()
        if len(counts) < len(CLASSES) or min(n for _, n in counts) < 10:
            print(f"{STRATIFY_BY}={value}: too few events {counts}, skipped")
            continue
        verdict = make_run(part).separability(**SEPARABILITY).frames["summary"]
        print(f"{STRATIFY_BY}={value}: {counts}")
        print(verdict[["cv_balanced_accuracy", "perm_p_value", "verdict"]].to_string(index=False))

In [ ]:
# The curated figures of every analysis run so far (the same ones the report shows)
for name, figures in run.figures().items():
    for title, fig in figures:
        print(f"{name}: {title}")
        display(fig)
        plt.close(fig)  # already displayed; stops the inline backend showing it twice

## 10. Save, report, dashboard

- `run.save()` writes every result as parquet plus a manifest.
- `run.report()` writes one self-contained HTML file: the thing to send to
  people who will not open a notebook.
- Browse saved runs with `uv run tessa-dashboard outputs/runs`.

In [ ]:
run_dir = run.save(OUTPUT_DIR / "runs", name="anomaly_vs_normal")
report = run.report(OUTPUT_DIR / "anomaly_vs_normal_report.html", title="Fault run-ups vs normal operation")
print("saved run :", run_dir)
print("report    :", report)